In [ ]:
data = [
    ("i am a student", "je suis un etudiant"),
    ("how are you", "comment allez vous"),
    ("i love machine learning", "j aime apprentissage automatique"),
    ("good morning", "bonjour"),
    ("thank you", "merci"),
    ("see you later", "a plus tard"),
    ("what is your name", "quel est votre nom"),
    ("where are you going", "ou allez vous"),
    ("i like coffee", "j aime le cafe"),
    ("welcome", "bienvenue")c:\Users\hares\Downloads\english_telugu_pairs_15000.txt
]

In [ ]:
#import necessary libraries
import tensorflow as tf
import numpy as np
from tensorflow.keras.layers import (
    TextVectorization,
    Embedding,
    Dense,
    LayerNormalization,
    MultiHeadAttention
)
from tensorflow.keras import Model

In [ ]:
#seperate input and output sentences
english_sentences = [x [0] for x in data] #list comprehension to extract English
#add start and end tokens
french_sentences = [
    "Start" + x[1] + "End"
    for x in data
]

In [ ]:
#tokenization
vocab_size = 1000 #keep a maximum of 1000 unique words in vocabulary
sequence_length = 20 #max length for each sentence

#tokenization for english sentences
source_vectorization = TextVectorization(
    max_tokens = vocab_size,
    output_mode = "int",
    output_sequence_length = sequence_length
)

#tokenization for french sentences
target_vectorization = TextVectorization(
    max_tokens = vocab_size,
    output_mode = "int",
    output_sequence_length = sequence_length
)

#this is where learning happens. (adapt the tokenizers to the data)
source_vectorization.adapt(
    english_sentences
)

target_vectorization.adapt(
    french_sentences
)

In [ ]:
#convert text into numbers
encoder_inputs = source_vectorization(english_sentences)
target_tokens = target_vectorization(french_sentences)

In [ ]:
#prepare decoder input and output
decoder_inputs = target_tokens[:, :-1]   #all tokens except the last one
decoder_targets = target_tokens[:, 1:]   #all tokens except the first one

In [ ]:
class PositionalEmbeddings(tf.keras.layers.Layer):

    def __init__(self,
                 sequence_length,
                 vocab_size,
                 embed_dim):

        super().__init__()

        self.token_embedding = Embedding(
            vocab_size,
            embed_dim
        )

        self.position_embedding = Embedding(
            sequence_length,
            embed_dim
        )

        self.sequence_length = sequence_length

    def call(self, inputs):

        length = tf.shape(inputs)[1]

        positions = tf.range(
            start=0,
            limit=length,
            delta=1
        )

        embedded_tokens = self.token_embedding(inputs)

        embedded_positions = self.position_embedding(
            positions
        )

        return embedded_tokens + embedded_positions

In [ ]:
#encoder block
#create a custom encoder layer
class TransformerEncoder(tf.keras.layers.Layer):
    #constructor
    #embed_dim:128, dense_dim:512, num_heads:4
    #num_heada:head1 --> grammar, head2 --> context, head3 --> relationships, head4 --> 
    def __init__(self,
             embed_dim,
             dense_dim,
             num_heads):
        super().__init__()

        self.attention = MultiHeadAttention(
            num_heads = num_heads,
            key_dim = embed_dim
        )

        #feed forward network
        #attentation mixes information
        #FNN learns comples features
        #i love ai (input) -->attention --> ai is related to love --> fnn --> ai is the opject being loved --> i love ai (output)
        self.dense_proj = tf.keras.Sequential([
            Dense(dense_dim,
                activation = "relu"
            ),
            Dense(embed_dim)
        ])
        #layer normalization
        self.layernorm1 = LayerNormalization()
        self.layernorm2 = LayerNormalization()

    def call(self, inputs):
        #self attention
        attention_output = self.attention(
            inputs, 
            inputs
        )
        #residual connection + layernorm
        proj_input = self.layernorm1(
            inputs + attention_output
        )
        #feed forward network
        proj_output = self.dense_proj(
            proj_input
        )
        #second residual connection + layernorm
        return self.layernorm2(
            proj_input + proj_output
        )

In [ ]:
#decoder block
class TransformerDecoder(tf.keras.layers.Layer):
    #constructor
    def __init__(self,
                 embed_dim,
                 dense_dim,
                 num_heads):
        super().__init__()

        self.self_attention = MultiHeadAttention(
            num_heads = num_heads,
            key_dim = embed_dim
        )

        self.cross_attention = MultiHeadAttention(
            num_heads = num_heads,
            key_dim = embed_dim
        )

        #feed forward network
        self.ffn = tf.keras.Sequential([
            Dense(
                dense_dim,
                activation = "relu"
            ),
            Dense(embed_dim)
        ])

        #layer normalization
        self.layernorm1 = LayerNormalization()
        self.layernorm2 = LayerNormalization()
        self.layernorm3 = LayerNormalization()

    def call(
            self,
            inputs,
            encoder_outputs
        ):
        #masked self attention
        attention1_output = self.self_attention(
            query = inputs,
            value = inputs,
            key = inputs,
            use_causal_mask= True
        )
        out1 = self.layernorm1(
            inputs + attention1_output
        )
        #cross attention
        attention2_output = self.cross_attention(
            out1,
            encoder_outputs
        )
        out2 = self.layernorm2(
            out1 + attention2_output
        )
        ffn_output = self.ffn(out2)

        return self.layernorm3(
            out2 + ffn_output
        )

In [ ]:
#build complete transformer
embed_dim = 32
dense_dim = 64
num_heads = 2
#encoder input layer
encoder_input = tf.keras.Input(
    shape = (None,),
    dtype = "int64"
)
x = PositionalEmbeddings(
    sequence_length,
    vocab_size,
    embed_dim
)(encoder_input)
#encoder block
encoder_output = TransformerEncoder(
    embed_dim,
    dense_dim,
    num_heads
)(x)
#decoder block
decoder_input = tf.keras.Input(
    shape = (None,),
    dtype = "int64"
)
x = PositionalEmbeddings(
    sequence_length,
    vocab_size,
    embed_dim
)(decoder_input)
decoder_output = TransformerDecoder(
    embed_dim,
    dense_dim,
    num_heads
)(x, encoder_output)
#final output layer
outputs = Dense(
    vocab_size,
    activation = "softmax"
)(decoder_output)
#build the model
transformer = Model(
    [encoder_input, decoder_input],
    outputs
)

In [ ]:
#compile and train the model
transformer.compile(
    optimizer = "adam",
    loss = "sparse_categorical_crossentropy",
    metrics = ["accuracy"]
)
transformer.fit(
    [encoder_inputs, decoder_inputs],
    decoder_targets,
    batch_size = 1,
    epochs = 25
)

Epoch 1/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 6s 12ms/step - accuracy: 0.9421 - loss: 0.2500
Epoch 2/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.9737 - loss: 0.1857
Epoch 3/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9632 - loss: 0.1259
Epoch 4/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9842 - loss: 0.0774
Epoch 5/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9895 - loss: 0.0489 
Epoch 6/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9947 - loss: 0.0348 
Epoch 7/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9895 - loss: 0.0243
Epoch 8/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9947 - loss: 0.0213 
Epoch 9/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9947 - loss: 0.0190
Epoch 10/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9947 - loss: 0.0165
Epoch 11/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9895 - loss: 0.0159
Epoch 12/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9895

In [ ]:
accuracy, loss = transformer.evaluate(
    [encoder_inputs, decoder_inputs],
    decoder_targets
)
print(f"Accuracy: {accuracy}, Loss: {loss}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 613ms/step - accuracy: 0.9947 - loss: 0.0096
Accuracy: 0.009609190747141838, Loss: 0.9947368502616882
